# LIME Explainability -- AQI Prediction Models

Explains the models actually produced by the last `main_train.py` run, loaded from `training_results.pkl`.

Unlike SHAP (`shap_explainability.ipynb`), LIME is model-agnostic meaning it works the same way regardless of whether the horizon's winner is Ridge or Random Forest.

LIME is also a local method and it explains one prediction at a time. There's no global summary plot the way SHAP has a beeswarm. The "summary" plot here is an approximation built by averaging feature weights across a sample of test rows.

In [ ]:
import sys
sys.path.append("..")

import joblib
import matplotlib.pyplot as plt

import os
os.chdir("C:\\Users\\abuhu\\Desktop\\Internship\\AQI-Pearls-Predictor")
print(os.getcwd())

from lime_explain import (
    explain_horizon,
    build_explainer,
    explain_instance,
    plot_local_explanation,
    compute_aggregate_importance,
    plot_aggregate_importance,
)

## Load training results

Same pickle the SHAP notebook.

In [ ]:
training_results = joblib.load("training_results.pkl")

for h, r in training_results.items():
    if r is None:
        print(f"{h}h: training was skipped (insufficient samples)")
        continue
    has_x_train = "X_train" in r
    print(f"{h}h: winner={r['winner_name']}, test_rmse={r['test_metrics']['rmse']:.3f}, X_train present={has_x_train}")

## Approximate-global + local LIME plots per horizon

For each horizon: an aggregate feature-importance plot (mean |LIME weight| across a sample of test rows), plus per-instance explanations for the best- and worst-predicted test rows.

In [ ]:
all_explanations = {}

for h, result in training_results.items():
    explanation = explain_horizon(result)
    all_explanations[h] = explanation

    if explanation is None:
        print(f"--- {h}h: skipped (training skipped, or X_train missing) ---")
        continue

    print(f"--- {h}h horizon ---")
    for name, fig in explanation.items():
        fig.suptitle(f"{h}h horizon — LIME {name}", y=1.02)
        plt.show()

## Save figures to disk

Optional

In [ ]:
import os

os.makedirs("LIME_FIGURES", exist_ok=True)

for h, explanation in all_explanations.items():
    if explanation is None:
        continue
    for name, fig in explanation.items():
        fig.savefig(f"LIME_FIGURES/lime_{h}h_{name}.png", bbox_inches="tight", dpi=150)

print("Saved LIME figures.")

## Explain an arbitrary single instance

Pick any row from the test set to inspect in detail. Useful for pulling out a specific hazardous-AQI case for the report, rather than only ever looking at the best/worst-error rows.

In [ ]:
HORIZON_TO_INSPECT = 24  # change as needed
ROW_INDEX = 0  # change to any row index within that horizon's X_test

result = training_results[HORIZON_TO_INSPECT]

if result is None or "X_train" not in result:
    print(f"{HORIZON_TO_INSPECT}h: not available (training skipped, or X_train missing).")
else:
    explainer = build_explainer(result["X_train"])
    row = result["X_test"].iloc[ROW_INDEX]
    true_value = result["y_test"].iloc[ROW_INDEX]
    predicted_value = result["winner_model"].predict(result["X_test"].iloc[[ROW_INDEX]])[0]

    print(f"True AQI: {true_value:.1f}, Predicted AQI: {predicted_value:.1f}")

    exp = explain_instance(explainer, result["winner_model"], row)
    fig = plot_local_explanation(exp, title_suffix=f" (row {ROW_INDEX}, {HORIZON_TO_INSPECT}h horizon)")
    plt.show()